In [ ]:
!pip install qiskit
!pip install qiskit-aer
!pip install qiskit-ibm-runtime

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.3/9.3 MB 29.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 34.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/54.5 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 45.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 19.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 386.8/386.8 kB 27.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.5/102.5 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.8/212.8 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.8/75.8 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 130.2/130.2 kB 7.2 MB/s eta 0:00:00


In [ ]:
import numpy as np
from numpy import pi
import random
from functools import partial
import matplotlib.pyplot as plt
from scipy.optimize import minimize
from scipy.linalg import eigh, sqrtm
from scipy.special import erf
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister, transpile
from qiskit.circuit import Parameter
from qiskit.circuit.classical import expr
from qiskit.quantum_info import SparsePauliOp, Statevector, Operator, random_statevector, process_fidelity
from qiskit.circuit.library import QAOAAnsatz, hamiltonian_variational_ansatz, XXPlusYYGate, CPhaseGate, UnitaryGate, U3Gate, CXGate,PauliEvolutionGate
from qiskit.synthesis import TwoQubitBasisDecomposer
from qiskit.synthesis.evolution import SuzukiTrotter
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit_aer import AerSimulator
from qiskit_ibm_runtime import SamplerV2 as Sampler, EstimatorV2 as Estimator,QiskitRuntimeService
from qiskit_ibm_runtime.options import EnvironmentOptions, EstimatorOptions,SamplerOptions

/usr/local/lib/python3.12/dist-packages/samplomatic/__init__.py:20: UserWarning: 
You have imported samplomatic==0.18.0 which is in 
beta development. Please expect breaking changes between 
minor versions and pin your dependencies accordingly.
  _warn_once_per_version(


## Measurement


In [ ]:
def measure_ZZ(qc, q0, q1, cbit):
    qc.cx(q0, q1)
    qc.measure(q1, cbit)
    qc.cx(q0, q1)

In [ ]:
def measure_XI(qc, q0, q1, cbit):
    qc.h(q0)
    qc.measure(q0, cbit)
    qc.h(q0)

**Important: fix convention $ Y = S X S^\dagger$. Later formulas must use the same convention to ensure a correct pauli tracking**

In [ ]:
def measure_YI(qc, q0, q1, cbit):
    qc.sdg(q0)
    measure_XI(qc, q0, q1, cbit)
    qc.s(q0)

In [ ]:
def measure_ZY(qc, q0, q1, cbit):

    # with the same convention, Y = S X S^d = S H Z H S^d
    qc.sdg(q1)
    qc.h(q1)
    measure_ZZ(qc, q0, q1, cbit)
    qc.h(q1)
    qc.s(q1)

In [ ]:
def measure_ZX(qc, q0, q1, cbit):

    qc.h(q1)
    measure_ZZ(qc, q0, q1, cbit)
    qc.h(q1)

:## Single qubit clifford group

In [ ]:
def add_H(qc, d, a, cbit):
    c = cbit

    measure_XI(qc, a, d, c[0])
    measure_ZY(qc, a, d, c[1])
    measure_YI(qc, a, d, c[2])
    measure_XI(qc, a, d, c[3])

    parity = expr.bit_xor(expr.bit_xor(c[0], c[1]), c[2])
    with qc.if_test(expr.logic_not(parity)):
        qc.y(d)

    qc.x(d)

    qc.reset(a) # easy to check with gate-based circuit

    return qc

In [ ]:
def add_S(qc, d, a, cbit):
    c = cbit

    # (ancilla, data) = (q+1, q)

    measure_XI(qc, a, d, c[0])
    measure_ZZ(qc, a, d, c[1])
    measure_YI(qc, a, d, c[2])
    measure_XI(qc, a, d, c[3])

    # measurement bit c_i encodes s_i = (-1)^{c_i}
    # s_i s_j = (-1)^{c[i] + c[j]}
    # product becomes XOR

    # Z^{(1 + s0 s1 s2)/2}
    # exponent = 1 when s0 s1 s2 = +1
    # s0 s1 s2 = (-1)^{c0 + c1 + c2}
    # +1 when (c0 + c1 + c2) mod 2 = 0 (even parity)

    parity = expr.bit_xor(expr.bit_xor(c[0], c[1]), c[2])
    with qc.if_test(expr.logic_not(parity)):
        qc.z(d)

    qc.reset(a) # easy to check with gate-based circuit

    return qc

In [ ]:
def add_SH(qc, d, a, cbit):
    c = cbit

    measure_XI(qc, a, d, c[0])  # s0
    measure_ZZ(qc, a, d, c[1])  # s1
    measure_ZY(qc, a, d, c[2])  # s2
    measure_YI(qc, a, d, c[3])  # s3
    measure_XI(qc, a, d, c[4])  # s4

    # X^{(1 + s0 s2 s3)/2} even parity
    parity_023 = expr.bit_xor(expr.bit_xor(c[0], c[2]), c[3])
    with qc.if_test(parity_023):
        qc.y(d)

    # Z^{(1 - s1 s2)/2} even parity
    parity_12 = expr.bit_xor(c[1], c[2])
    with qc.if_test(parity_12):
        qc.z(d)

    qc.reset(a)

    return qc

In [ ]:
def add_HSH(qc, d, a, cbit):
    c = cbit

    measure_XI(qc, a, d, c[0])
    measure_ZZ(qc, a, d, c[1])
    measure_ZY(qc, a, d, c[2])
    measure_XI(qc, a, d, c[3])

    # Y^{(1 - s0 s3)/2} odd parity
    parity_03 = expr.bit_xor(c[0], c[3])
    with qc.if_test(parity_03):
        qc.y(d)

    # X^{(1 + s1 s2)/2} even parity
    parity_12 = expr.bit_xor(c[1], c[2])
    with qc.if_test(expr.logic_not(parity_12)):
        qc.x(d)

    qc.reset(a)

    return qc

In [ ]:
def add_HS(qc, d, a, cbit):
    c = cbit

    measure_XI(qc, a, d, c[0])  # s0
    measure_ZY(qc, a, d, c[1])  # s1
    measure_ZZ(qc, a, d, c[2])  # s2
    measure_YI(qc, a, d, c[3])  # s3
    measure_XI(qc, a, d, c[4])  # s4

    # X^{(1 + s1 s2)/2} even parity
    parity_12 = expr.bit_xor(c[1], c[2])
    with qc.if_test(expr.logic_not(parity_12)):
        qc.x(d)

    # Z^{(1 + s0 s1 s3)/2} even parity
    parity_013 = expr.bit_xor(expr.bit_xor(c[0], c[1]), c[3])
    with qc.if_test(expr.logic_not(parity_013)):
        qc.z(d)

    qc.reset(a)

    return qc

In [ ]:
def sqrt_W():
    X = np.array([
        [0, 1],
        [1, 0]
    ], dtype=complex)

    Y = np.array([
        [0, -1j],
        [1j, 0]
    ], dtype=complex)

    W = (X + Y) / np.sqrt(2)

    return sqrtm(W)

## Single-qubit gate in supremacy circuit

In [ ]:
def add_sqrtX(qc, d, a, cbit):
    qc = add_HSH(qc, d, a, cbit)
    return qc

In [ ]:
def add_sqrtY(qc, d, a, cbit):
   qc = add_S(qc, d, a, cbit)
   qc = add_HS(qc, d, a, cbit)
   return qc

In [ ]:
def add_sqrtW(qc, d, a, cbit):
    qc.tdg(d)
    qc = add_sqrtX(qc, d, a, cbit)
    qc.t(d)
    return qc

## Two-qubit gate in supremacy circuit

In [ ]:
# fSim matrix

def fSim(theta, phi):
    return np.array([
        [1, 0, 0, 0],
        [0, np.cos(theta), -1j*np.sin(theta), 0],
        [0, -1j*np.sin(theta), np.cos(theta), 0],
        [0, 0, 0, np.exp(-1j*phi)]
    ], dtype=complex)

In [ ]:
def add_CNOT(qc, c, a, t, cbit):
    # needs 8 cbits. all_ reuses cbit[0-4]. measure_ uses cbit[5-7]
    # Qubit A is initialized in an eigenstate of Z

    # prepare a in z-basis
    qc.reset(a)

    measure_ZX(qc, c, a, cbit[5])
    measure_ZX(qc, a, t, cbit[6])

    # this is single X-measurement on qubit a. h is on level of simulation, not circuit element
    qc.h(a)
    qc.measure(a, cbit[7])
    qc.h(a)

    # reset a to 0 for an easier comparision
    qc.reset(a)

    # we use cbit directly because it stores as 0(even) and 1(odd), as needed in paper
    P1 = cbit[5]
    P2 = cbit[6]
    M  = cbit[7]

    # X_t^{(P1 ⊕ M)}
    xt = expr.bit_xor(P1, M)
    with qc.if_test(xt):
        qc.x(t)

    #Z_c^{P2}
    with qc.if_test((P2, 1)):
        qc.z(c)

    return qc

In [ ]:
def apply_rz_or_s(qc, qubit, angle):

    # identify z-rotation as clifford gate

    # normalize to [-π, π]
    theta = (angle + np.pi) % (2*np.pi) - np.pi

    # check multiple of π/2
    k = round(theta / (np.pi/2))

    if abs(theta - k*(np.pi/2)) < 1e-3:
        k_mod = k % 4
        for _ in range(k_mod):
            qc.s(qubit)
    else:
        qc.rz(theta, qubit)

    return qc

In [ ]:
def get_zsx_blocks_and_cx(theta, phi):

    # decompose fSim into 3 CNOTs and Euler rotations in ZSX basis

    qc = QuantumCircuit(2)
    qc.append(UnitaryGate(fSim(theta, phi)), [0, 1])

    U_target = Operator(qc).data

    decomposer = TwoQubitBasisDecomposer(
        CXGate(),
        euler_basis="ZSX",
    )

    synth = decomposer(U_target)

    synth = transpile(
        synth,
        basis_gates=["rz", "sx", "x", "cx"],
        optimization_level=3,
    )

    blocks = []
    cx_dirs = []
    current = {0: [], 1: []}

    for instr in synth.data:
        inst = instr.operation
        qargs = instr.qubits
        name = inst.name

        if name == "cx":
            blocks.append(current)
            current = {0: [], 1: []}

            ctrl = synth.find_bit(qargs[0]).index
            targ = synth.find_bit(qargs[1]).index
            cx_dirs.append((ctrl, targ))

            continue

        q = synth.find_bit(qargs[0]).index

        if name == "rz":
            current[q].append(("rz", float(inst.params[0])))
        elif name == "sx":
            current[q].append(("sx", None))
        elif name == "x":
            current[q].append(("x", None))

    blocks.append(current)

    return blocks, cx_dirs

In [ ]:
def add_TwoQubitGate(qc, c, a, t, theta, phi, Z1, Z2, Z3, Z4, cbit):

    # build two qubit gate for supremacy circuit

    qc.rz(Z1, c)
    qc.rz(Z2, t)

    blocks, cx_dirs = get_zsx_blocks_and_cx(theta, phi)

    for layer in range(4):

        block = blocks[layer]

        # ---- 1Q gates ----
        for qubit in [0, 1]:

            target = c if qubit == 0 else t

            for name, angle in block[qubit]:

                if name == "rz":
                    qc = apply_rz_or_s(qc, target, angle)

                elif name == "sx":
                    qc = add_sqrtX(qc, target, a, cbit)

                elif name == "x":
                    qc = add_sqrtX(qc, target, a, cbit)
                    qc = add_sqrtX(qc, target, a, cbit)

        # ---- CX with correct direction ----
        if layer < 3:
            ctrl, targ = cx_dirs[layer]

            if ctrl == 0:
                qc = add_CNOT(qc, c, a, t, cbit)
            else:
                qc = add_CNOT(qc, t, a, c, cbit)

    qc.rz(Z3, c)
    qc.rz(Z4, t)

    return qc

# 12-Qubit Supremacy Circuit

We use only **1 ancilla qubit**.  
This **12+1** construction is equivalent to the original **12+12 tetron layout** due to:

- ancilla qubits form a product state instantanouesly in **noiseless** condition (*correct physics*),
- and successful MVP circuit test *(correct implementation)*.

The initial state is the classical bitstring: $|00\cdots0\rangle$

The circuit data is stored in: **circuit_n12_m14_s0_e0_pEFGH_snapped.json**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os

Mounted at /content/drive


In [ ]:
import json

with open("/content/drive/My Drive/microsoft/circuit_n12_m14_s0_e0_pEFGH_snapped.json", "r") as f:
    data = json.load(f)

# ==========================================
# 1. Two-qubit gate summary
# ==========================================

pair_to_layer = {}

for layer, pairs in data["efgh_tetron_pairs"].items():
    for pair in pairs:
        pair_to_layer[tuple(pair)] = layer

two_qubit_summary = []
seen = set()

for moment in data["moments"]:

    for gate in moment["gates"]:

        if gate["gate"] != "FSimGate":
            continue

        pair = tuple(gate["qubits"])

        # parameters repeat across cycles
        if pair in seen:
            continue

        seen.add(pair)

        two_qubit_summary.append((
            ("layer", pair_to_layer[pair]),
            ("pair", pair),
            ("theta", gate["theta"]),
            ("phi", gate["phi"]),
            ("theta_original", gate["theta_original"]),
            ("phi_original", gate["phi_original"]),
        ))

# ==========================================
# 2. Single-qubit gate summary
# ==========================================

single_qubit_summary = []

for moment in data["moments"]:

    layer = []

    for gate in moment["gates"]:

        gate_name = gate["gate"]

        # keep only single-qubit gates
        if gate_name not in ["sqrtX", "sqrtY", "sqrtW"]:
            continue

        q = gate["qubits"][0]

        # rename to your notation
        if gate_name == "sqrtX":
            label = "sX"

        elif gate_name == "sqrtY":
            label = "sY"

        else:
            label = "sW"

        layer.append((
            ("qubit", q),
            ("gate", label),
        ))

    # only keep non-empty layers
    if layer:
        single_qubit_summary.append(layer)

In [ ]:
def Supremacy_MBQC_trick(qc, single_qubit_summary, two_qubit_summary):

    ordered_layers = ["E", "F", "G", "H"]
    cycle = len(single_qubit_summary)

    if not qc.cregs:
        qc.add_register(ClassicalRegister(8))
    cbit = qc.cregs[0]

    N = len(single_qubit_summary[0])
    anc = N

    for p in range(cycle):

        # =========================
        # 1. Single-qubit layer
        # =========================
        single_layer = single_qubit_summary[p]
        for item in single_layer:
            entry = dict(item)
            q = entry["qubit"] - 1
            gate = entry["gate"]

            if gate == "sX":
                qc = add_sqrtX(qc, q, anc, cbit)
            elif gate == "sY":
                qc = add_sqrtY(qc, q, anc, cbit)
            else:
                qc = add_sqrtW(qc, q, anc, cbit)
        qc.barrier()

        # final layer is single-qubit only
        if p == cycle - 1:
            continue

        # =========================
        # 2. Two-qubit layer
        # =========================
        for layer in ordered_layers:
            for item in two_qubit_summary:
                item_layer = dict(item)["layer"]
                if item_layer != layer:
                    continue

                pair = dict(item)["pair"]
                theta = dict(item)["theta"]
                phi = dict(item)["phi"]
                c, t = pair
                c -= 1
                t -= 1
                Z1 = 0
                Z2 = 0
                Z3 = 0
                Z4 = 0

                qc = add_TwoQubitGate(qc, c, anc, t, theta, phi, Z1, Z2, Z3, Z4, cbit)
            qc.barrier()

    return qc

In [ ]:
def Supremacy_gate(qc, single_qubit_summary, two_qubit_summary):

    ordered_layers = ["E", "F", "G", "H"]
    cycle = len(single_qubit_summary)

    for p in range(cycle):

        # =========================
        # 1. Single-qubit layer
        # =========================
        single_layer = single_qubit_summary[p]
        for item in single_layer:
            entry = dict(item)
            q = entry["qubit"] - 1
            gate = entry["gate"]

            if gate == "sX":
                qc.sx(q)
            elif gate == "sY":
                qc.ry(np.pi / 2, q)
            else:
                qc.append(UnitaryGate(sqrt_W()), [q])
        qc.barrier()

        # final layer is single-qubit only
        if p == cycle - 1:
            continue

        # =========================
        # 2. Two-qubit layer
        # =========================
        for layer in ordered_layers:
            for item in two_qubit_summary:
                entry = dict(item)
                if entry["layer"] != layer:
                    continue

                c, t = entry["pair"]
                # Google indexing -> Qiskit indexing
                c -= 1
                t -= 1
                theta = entry["theta"]
                phi = entry["phi"]
                Z1 = 0
                Z2 = 0
                Z3 = 0
                Z4 = 0

                qc.rz(Z1, c)
                qc.rz(Z2, t)
                qc.append(UnitaryGate(fSim(theta, phi)), [c, t])
                qc.rz(Z3, c)
                qc.rz(Z4, t)
            qc.barrier()

    return qc

In [ ]:
N=12
seed = 0
qc = QuantumCircuit(N)
cycle = 15
qc = Supremacy_gate(qc, single_qubit_summary, two_qubit_summary)
qc.draw(fold=-1)

┌─────────┐ ░ ┌───────┐┌──────────┐┌───────┐ ░ ┌───────┐                        ┌──────────┐ ┌───────┐                                    ░ ┌───────┐┌──────────┐ ┌───────┐            ░ ┌───────┐┌──────────┐ ┌───────┐            ░ ┌─────────┐ ░ ┌───────┐┌──────────┐┌───────┐ ░ ┌───────┐                        ┌──────────┐ ┌───────┐                                    ░ ┌───────┐┌──────────┐ ┌───────┐            ░ ┌───────┐┌──────────┐ ┌───────┐            ░    ┌────┐   ░ ┌───────┐┌──────────┐┌───────┐ ░ ┌───────┐                        ┌──────────┐ ┌───────┐                                    ░ ┌───────┐┌──────────┐ ┌───────┐            ░ ┌───────┐┌──────────┐ ┌───────┐            ░ ┌─────────┐ ░ ┌───────┐┌──────────┐┌───────┐ ░ ┌───────┐                        ┌──────────┐ ┌───────┐                                    ░ ┌───────┐┌──────────┐ ┌───────┐            ░ ┌───────┐┌──────────┐ ┌───────┐            ░    ┌────┐   ░ ┌───────┐┌──────────┐┌───────┐ ░ ┌───────┐                        ┌──────────┐ ┌───────┐                                    ░ ┌───────┐┌──────────┐ ┌───────┐            ░ ┌───────┐┌──────────┐ ┌───────┐            ░ ┌─────────┐ ░ ┌───────┐┌──────────┐┌───────┐ ░ ┌───────┐                        ┌──────────┐ ┌───────┐                                    ░ ┌───────┐┌──────────┐ ┌───────┐            ░ ┌───────┐┌──────────┐ ┌───────┐            ░ ┌─────────┐ ░ ┌───────┐┌──────────┐┌───────┐ ░ ┌───────┐                        ┌──────────┐ ┌───────┐                                    ░ ┌───────┐┌──────────┐ ┌───────┐            ░ ┌───────┐┌──────────┐ ┌───────┐            ░    ┌────┐   ░ ┌───────┐┌──────────┐┌───────┐ ░ ┌───────┐                        ┌──────────┐ ┌───────┐                                    ░ ┌───────┐┌──────────┐ ┌───────┐            ░ ┌───────┐┌──────────┐ ┌───────┐            ░ ┌─────────┐ ░ ┌───────┐┌──────────┐┌───────┐ ░ ┌───────┐                        ┌──────────┐ ┌───────┐                                    ░ ┌───────┐┌──────────┐ ┌───────┐            ░ ┌───────┐┌──────────┐ ┌───────┐            ░ ┌─────────┐ ░ ┌───────┐┌──────────┐┌───────┐ ░ ┌───────┐                        ┌──────────┐ ┌───────┐                                    ░ ┌───────┐┌──────────┐ ┌───────┐            ░ ┌───────┐┌──────────┐ ┌───────┐            ░    ┌────┐   ░ ┌───────┐┌──────────┐┌───────┐ ░ ┌───────┐                        ┌──────────┐ ┌───────┐                                    ░ ┌───────┐┌──────────┐ ┌───────┐            ░ ┌───────┐┌──────────┐ ┌───────┐            ░ ┌─────────┐ ░ ┌───────┐┌──────────┐┌───────┐ ░ ┌───────┐                        ┌──────────┐ ┌───────┐                                    ░ ┌───────┐┌──────────┐ ┌───────┐            ░ ┌───────┐┌──────────┐ ┌───────┐            ░    ┌────┐   ░ ┌───────┐┌──────────┐┌───────┐ ░ ┌───────┐                        ┌──────────┐ ┌───────┐                                    ░ ┌───────┐┌──────────┐ ┌───────┐            ░ ┌───────┐┌──────────┐ ┌───────┐            ░ ┌─────────┐ ░ ┌───────┐┌──────────┐┌───────┐ ░ ┌───────┐                        ┌──────────┐ ┌───────┐                                    ░ ┌───────┐┌──────────┐ ┌───────┐            ░ ┌───────┐┌──────────┐ ┌───────┐            ░ ┌─────────┐ ░ 
 q_0: ┤ Ry(π/2) ├─░─┤ Rz(0) ├┤0         ├┤ Rz(0) ├─░─┤ Rz(0) ├────────────────────────┤1         ├─┤ Rz(0) ├────────────────────────────────────░─┤ Rz(0) ├┤0         ├─┤ Rz(0) ├────────────░─┤ Rz(0) ├┤1         ├─┤ Rz(0) ├────────────░─┤ Unitary ├─░─┤ Rz(0) ├┤0         ├┤ Rz(0) ├─░─┤ Rz(0) ├────────────────────────┤1         ├─┤ Rz(0) ├────────────────────────────────────░─┤ Rz(0) ├┤0         ├─┤ Rz(0) ├────────────░─┤ Rz(0) ├┤1         ├─┤ Rz(0) ├────────────░────┤ √X ├───░─┤ Rz(0) ├┤0         ├┤ Rz(0) ├─░─┤ Rz(0) ├────────────────────────┤1         ├─┤ Rz(0) ├────────────────────────────────────░─┤ Rz(0) ├┤0         ├─┤ Rz(0) ├────────────░─┤ Rz(0) ├┤1         ├─┤ Rz(0) ├────────────░─┤ Unitary ├─░─┤ Rz(0) ├┤0         ├┤ Rz(0) ├─░─┤ Rz(0) ├─────────────

In [ ]:
sim = AerSimulator(method="statevector")

N = 12
tol = 1e-6

    # --- direct circuit ---
qc1 = QuantumCircuit(N+1)
qc1 = Supremacy_gate(qc1, single_qubit_summary, two_qubit_summary)
sv1 = Statevector.from_instruction(qc1)

    # --- MBQC circuit ---
qc2 = QuantumCircuit(N+1)
qc2 = Supremacy_MBQC_trick(qc2, single_qubit_summary, two_qubit_summary)
qc2.save_statevector(conditional=True)
result = sim.run(qc2, shots=1).result()
data = result.data(0)["statevector"]
sv_mbqc = list(data.values())[0]

overlap = abs(np.vdot(sv1.data, sv_mbqc.data))

if abs(overlap - 1) < tol:
    print(f"✅ {N}+1 supremacy circuit pass the test "
          f"(overlap ≈ 1 within tolerance {tol})")
else:
    print(f"❌ Test failed: overlap = {overlap}")

✅ 12+1 supremacy circuit pass the test (overlap ≈ 1 within tolerance 1e-06)
